In [ ]:
# Imports: 
#Imports
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import pandas as pd
import scipy.stats as stats 
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import tkinter as tk
from tkinter import filedialog
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

In [ ]:
# Call in files

boundary_txt = f""
interior_txt = f""
full_txt=f""


In [ ]:
# Single gaussian fns
def fit_gaussian_from_dataset(x_data, y_data):
    """
    Fit a Gaussian curve to the provided dataset (x_data, y_data).
    Args:
    - x_data: Array-like, x values of the data
    - y_data: Array-like, y values of the data
    
    Returns:
    - A_fit: Amplitude of the fitted Gaussian
    - mu_fit: Mean (center) of the fitted Gaussian
    - sigma_fit: Standard deviation (width) of the fitted Gaussian
    - popt: Optimal parameters from curve fitting
    - pcov: Covariance matrix
    """
    # Initial guess for the parameters [A, mu, sigma]
    initial_guess = [max(y_data), np.mean(x_data), np.std(x_data)]

    try:
        # Fit the Gaussian curve to the data
        popt, pcov = curve_fit(gaussian, x_data, y_data, p0=initial_guess, maxfev=100000000)
        
        # Extract fitted parameters
        A_fit, mu_fit, sigma_fit = popt
        min_raw=np.min(x_data)
        mean_raw=np.mean(x_data)
        max_raw=np.max(x_data)
        # Print the fitted parameters for feedback
        #print(f"Fitted parameters:\nAmplitude: {A_fit}\nMean: {mu_fit}\nSigma: {sigma_fit}")
        #print(f"Raw data parameters are:\n Min = {min_raw}\n Mean = {mean_raw}\n Max = {max_raw}")
       
        # Return the fitted parameters and covariance matrix
        return A_fit, mu_fit, sigma_fit, popt, pcov

    except Exception as e:
        print(f"Error fitting Gaussian: {e}")
        return None, None, None, None, None
    


# --- Gaussian fitting---
def fit_gaussian (file_path, rows):
    """Performs Gaussian fitting on data from a given file path."""
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        data = np.loadtxt(f, skiprows=rows)
        
    x_data = data[:, 0]
    y_data = data[:, 1]
    
    # Estimate initial guess and peak count (Assumes estimate_guess_from_data is defined) 
    #for multigaussian
    x_fit= np.linspace(np.min(x_data), np.max(x_data), 1000000)
    '''#initial_guess, n_peaks = estimate_guess_from_data(x_data, y_data)
    #popt, _ = curve_fit(multi_gaussian, x_data, y_data, p0=initial_guess, maxfev=100000000)
    #y_fit=multi_gaussian(x_fit,*popt)
    # Extract Gaussian parameters
    gaussians = []
    for i in range(len(popt) // 3):
        amp = popt[i*3]
        cen = popt[i*3+1]
        wid = abs(popt[i*3+2])
        gaussians.append({'amp': amp, 'cen': cen, 'wid': wid})

    highest = gaussians[np.argmax([g['amp'] for g in gaussians])]
    fwhm = 2.355 * highest['wid']'''

    #For single gaussian: 
    A, mu, sigma, popt, pcov = fit_gaussian_from_dataset(x_data, y_data)
    y_fit=gaussian(x_fit,*popt)
    
    peak_index_max=np.argmax(y_fit)
    peak_centre=x_fit[peak_index_max]
    
    # Extract Gaussian parameters
    gaussians = []
    for i in range(len(popt) // 3):
        amp = popt[i*3]
        cen = popt[i*3+1]
        wid = abs(popt[i*3+2])
        gaussians.append({'amp': amp, 'cen': cen, 'wid': wid})

    highest = gaussians[np.argmax([g['amp'] for g in gaussians])]
    fwhm = 2.355 * highest['wid']
    
    return x_data, y_data, x_fit, y_fit, peak_centre, fwhm, popt, pcov

In [ ]:
# Multi-gaussian def + process:
# Function to model N Gaussians
def multi_gaussian(x, *params):
    n = len(params) // 3
    y = np.zeros_like(x)
    for i in range(n):
        amp = params[i * 3]
        cen = params[i * 3 + 1]
        wid = params[i * 3 + 2]
        y += amp * np.exp(-((x - cen) ** 2) / (2 * wid ** 2))
    return y

# Auto-generate initial guess based on data
def estimate_guess_from_data(x, y, prominence=0.1, distance=10):
    y_smooth = gaussian_filter1d(y, sigma=2)
    peaks, _ = find_peaks(y_smooth, prominence=prominence, distance=distance)
    
    # Sort peaks by height (amplitude) and limit to top 2
    if len(peaks) > 2:
        sorted_indices = np.argsort(properties['prominences'])[::-1][:2]
        peaks = peaks[sorted_indices]

    guess = []
    for i in peaks:
        amp = y[i]
        cen = x[i]
        wid = 0.001 # Adjust based on data
        guess += [amp, cen, wid]
    
    return guess, len(peaks)

#Multi-gaussian fitting for all files within file directory
def multi_gaussian_folder(file_path, rows, linecolour, name):

    #Plotting parameters
    dpi_setting = 600
    figsize_x, figsize_y = (2227/dpi_setting), (1498/dpi_setting)
    tick_length = 6
    legend_font = 12
    base_font_size = 12
    scatter_size = 6
    line_width = 1.5
    fig, ax = plt.subplots(1, 1, sharey=True, figsize=(figsize_x, figsize_y))
    plt.rcParams.update({'font.size': base_font_size})
    ax = plt.gca()
    ax.tick_params(direction='in', length=tick_length)
    ax.tick_params(axis='both')
    ax.margins (0.5,0.5)
    plt.xlabel("SPV (V)")
    ax.autoscale(enable=None, axis="x", tight=True)
    plt.ylabel("Distribution (V$^{-1}$)")
    # Limit number of ticks
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 ticks on x-axis
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 ticks on y-axis
    results = []
    #print(f"\nProcessing file: {filename}") #Use if you want each individual file processed

    data = np.loadtxt(file_path, skiprows=rows)

    x_data = data[:, 0]
    y_data = data[:, 1]
            
    # Estimate initial guess and peak count
    initial_guess, n_peaks = estimate_guess_from_data(x_data, y_data)
    min_raw=np.min(x_data)
    max_raw=np.max(x_data)
    mean_raw=np.mean(x_data)
    #x_fit= np.linspace(np.min(x_data), np.max(x_data), 1000000)
    popt, pcov = curve_fit(multi_gaussian, x_data, y_data, p0=initial_guess, maxfev=100000000)
    y_fit=multi_gaussian(x_data,*popt)
    peak_index_max=np.argmax(y_fit)
    x_peak_centre=x_data[peak_index_max]
    perr=np.sqrt(np.diag(pcov))
    #For finding min and max *i.e. either side of the peaks. 
    # Extract Gaussian parameters
    gaussians = []
    for i in range(len(popt) // 3):
        amp = popt[i*3]
        cen = popt[i*3+1]
        wid = abs(popt[i*3+2])
        gaussians.append({'amp': amp, 'cen': cen, 'wid': wid})

    # Identify the highest peak
    #highest = max(gaussians, key=lambda g: g['amp'])
    highest = gaussians[np.argmax([g['amp'] for g in gaussians])]
# FWHM = 2.355 * sigma
    sigma=highest['wid']
    fwhm = 2.355 *sigma
    x_left = highest['cen'] - fwhm / 2
    x_right = highest['cen'] + fwhm / 2
    
    # Store in lists for output

    # Optional: print for debugging
    # 3. Create the summary text for the plot
    statistics=(f"Peak: {highest['cen']*1000:.3f} mV\n"
              f"FWHM: {fwhm*1000:.2f} mV\n"
              f"Sigma: {sigma*1000:.3f} mV"
              f"Bounds: [{x_left*1000:.2f}, {x_right*1000:.2f}]mV")
    print(statistics) 
    if n_peaks is not None:
   
        # Shading spread and peak centre
        ax.axvspan(x_left, x_right, color='gray', alpha=0.2, label='FWHM Region', zorder=0)
        ax.axvline(x=highest['cen'], color='black', linestyle='--', linewidth=line_width, label='Peak Center', zorder=4)
        #Raw data and fitted gaussian: 
        ax.scatter(x_data, y_data, label=f"name",color=linecolour, s=scatter_size, zorder=2)
        ax.plot(x_data, multi_gaussian(x_data, *popt), label=f"_nolegend_", color=linecolour,linewidth=line_width, zorder=3)
        #Labelling and layout adjustment
        '''ax.text(0.95,0.95,statistics, transform=ax.transAxes,
                verticalalignment='top', horizontalalignment='right', 
                facecolor='white', alpha=0.8, edgecolour=None,
                fontweight='bold', fontsize=10)'''
        ax.set_xlabel("CPD(V)")
        ax.set_ylabel("Distribution $V^{-1}$")
        ax.autoscale()
        #plt.legend()
        plt.tight_layout()
        #plt.show()
    y_fit = multi_gaussian(x_data, *popt)
    #plt.autoscale(enable=True, axis='y', tight=False)
    return x_data, y_data, y_fit, statistics, perr
    plt.show()    

In [ ]:
# Plotting for GB,GI+full:


In [ ]:
# Call in functions:


In [ ]:
# Call in plotting:


In [ ]:
# Pull out statistics: